# Fyre opp Neo4j
Sjekke at vi kan kjøre Neo4J

Starte med
```
sudo apt install podman
```

In [50]:
%%bash
PWD=$(pwd)
mkdir -p neo4j
podman run \
    -p 7474:7474 -p 7687:7687 \
    --userns=keep-id \
    -e NEO4J_dbms_security_procedures_unrestricted=apoc.* \
    -e NEO4J_dbms_security_procedures_allowlist=apoc.* \
    -e NEO4J_apoc_import_file_enabled=true \
    -v $PWD/neo4j/data:/data:Z \
    -v $PWD/neo4j/logs:/logs:Z \
    -v $PWD/neo4j/import:/import:Z \
    -v $PWD/neo4j/plugins:/plugins:Z \
    -e NEO4J_AUTH=neo4j/password \
    -e NEO4J_PLUGINS='["apoc"]' \
    -d docker.io/library/neo4j:latest

d1349c6d65b5e1ac85ac00826fbe43f4e4e51a6553dd897b456c304a2d69734d


Når man er ferdig

In [49]:
%%bash
podman kill c6b264b88de089d1c310a3042bdde143131c89a1e1c1006424d547936ee28a52

c6b264b88de089d1c310a3042bdde143131c89a1e1c1006424d547936ee28a52


Gi databasen litt tid til å starte

In [106]:
from neo4j import GraphDatabase

# 1. Define connection details
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Connection Successful!")

Connection Successful!


Sjekke at vi har APOC tilgjengelig

In [107]:
records, summary, keys = driver.execute_query(
    """RETURN apoc.version() AS version;""")

print(f"Server Address: {summary.server.address}")
print("Keys:")
for k in range(len(keys)):
    print(f"\t{keys[k]}: {records[k]}")
#

print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")



Server Address: 127.0.0.1:7687
Keys:
	version: <Record version='2025.11.2'>
Grafen
	Nye noder: 0
	Nye kanter: 0
Ressursbruk
	Kjøringen: 1ms
	Å konsumere: 1ms


Ofte vil vi lage ting i Networkx og så laste det inn slik at andre kan bruke det.

In [117]:
import gzip
import networkx as nx

with gzip.open("data/email.edgelist.txt.gz", "rt") as fd:
    G = nx.read_edgelist(fd, create_using=nx.DiGraph())
#
G.remove_edges_from(nx.selfloop_edges(G))
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

nx.set_node_attributes(G, ":Person", name="labels")

# Skriv ut grafen
nx.write_graphml(G, "neo4j/import/large_graph.graphml", named_key_ids=True)


Nodes: 57194
Edges: 103083


Les inn

In [118]:
records, summary, keys = driver.execute_query(
    """CALL apoc.import.graphml("large_graph.graphml", {storeNodeIds: true, readLabels: true})""")
for record in records:
    # Converts the entire record into a Python dict
    record_dict = record.data()
#
for k in keys:
    print(f"\t{k}: {record_dict[k]}")


print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	file: large_graph.graphml
	source: file
	format: graphml
	nodes: 57194
	relationships: 51542
	properties: 0
	time: 678
	rows: 0
	batchSize: -1
	batches: 0
	done: True
	data: None
Grafen
	Nye noder: 0
	Nye kanter: 0
Ressursbruk
	Kjøringen: 1ms
	Å konsumere: 680ms


Sjekke en velkjent node:

In [110]:
records, summary, keys = driver.execute_query(
    """MATCH (n:Person {ID:'0'})
RETURN n
""")
for record in records:
    # Converts the entire record into a Python dict
    record_dict = record.data()
#

print(keys)
print(records)

['n']
[]
